In [1]:
#IMPORTING Library
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.tree import DecisionTreeRegressor, DecisionTreeClassifier
from sklearn.metrics import mean_absolute_error, f1_score


In [2]:
#Dataset yang dibutuhkan (kecamatan)
jmlh_penduduk_kecamatan_path = "jmlh_penduduk_kecamatan.csv"
kecamatan_df_raw = pd.read_csv(jmlh_penduduk_kecamatan_path)
kecamatan_df = kecamatan_df_raw[kecamatan_df_raw['Provinsi'] != 'Indonesia'].dropna() #ini buat buang data yang ga perlu
kecamatan_df.head()

,Provinsi,Jumlah Kecamatan
0,Aceh,290
1,Sumatera Utara,455
2,Sumatera Barat,179
3,Riau,172
4,Jambi,144


In [3]:
#Dataset yang dibutuhkan (Provinsi)
jmlh_penduduk_provinsi_path = "jmlh_penduduk_provinsi.csv"
provinsi_df_raw = pd.read_csv(jmlh_penduduk_provinsi_path)
provinsi_df = provinsi_df_raw[:39] #ini buat ngestruktur ulang data yang perlu
provinsi_df.head()

,Provinsi,Jumlah Penduduk (Ribu),Laju Pertumbuhan Penduduk per Tahun,Persentase Penduduk,Kepadatan Penduduk per km persegi (Km2),Rasio Jenis Kelamin Penduduk
0,Aceh,5626.0,1.37,1.98,99.0,100.9
1,Sumatera Utara,15785.8,1.37,5.55,218.0,100.8
2,Sumatera Barat,5914.3,1.41,2.08,140.0,101.5
3,Riau,6811.2,1.34,2.39,76.0,104.3
4,Jambi,3768.5,1.28,1.32,77.0,103.2


In [58]:
#Penggabungan data kecamatan dan provinsi
df_final = pd.merge(kecamatan_df, provinsi_df)
df_final.columns

Index(['Provinsi', 'Jumlah Kecamatan', 'Jumlah Penduduk (Ribu)',
       'Laju Pertumbuhan Penduduk per Tahun', 'Persentase Penduduk',
       'Kepadatan Penduduk per km persegi (Km2)',
       'Rasio Jenis Kelamin Penduduk'],
      dtype='object')

In [59]:
# Threshold untuk menentukan hitungan strategis atau tidak suatu wilayah
threshold = df_final["Kepadatan Penduduk per km persegi (Km2)"].median()
df_final["Strategis"] = (df_final['Kepadatan Penduduk per km persegi (Km2)'] > threshold).astype(int) 
df_final.columns

Index(['Provinsi', 'Jumlah Kecamatan', 'Jumlah Penduduk (Ribu)',
       'Laju Pertumbuhan Penduduk per Tahun', 'Persentase Penduduk',
       'Kepadatan Penduduk per km persegi (Km2)',
       'Rasio Jenis Kelamin Penduduk', 'Strategis'],
      dtype='object')

In [60]:
#Feature and Target 

df_final_features =[
                    "Jumlah Kecamatan",
                    "Jumlah Penduduk (Ribu)",
                    "Laju Pertumbuhan Penduduk per Tahun",
                    "Persentase Penduduk"
                    ]
                    
X = df_final[df_final_features]
y = df_final['Strategis']


In [80]:
X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=42, stratify=y)
X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp)

df_model = {
    "Decision Tree Regression": DecisionTreeRegressor(max_depth=3, random_state=42),
    "Decision Tree Classifier": DecisionTreeClassifier(max_depth=3, random_state=42)
}

for name, model in df_model.items():
    model.fit(X_train, y_train)

    val_pred = model.predict(X_val)

    if "Regression" in name:
        mae = mean_absolute_error(y_val, val_pred)
        print(f"[{name}]")
        print(f"  > Mean Absolute Error (MAE): {mae:.4f}")
    else:
        scores = cross_val_score(model, X,y, cv=5, scoring='f1')
        f1 = f1_score(y_val, val_pred)
        print(f"Rata-rata F1-Score: {scores.mean() * 100:.2f}%")
        print(f"Standar Deviasi (Konsistensi): {scores.std():.4f}")



[Decision Tree Regression]
  > Mean Absolute Error (MAE): 0.3333
Rata-rata F1-Score: 55.67%
Standar Deviasi (Konsistensi): 0.1867
